# Unit 4: Adapt observations and actions

Convert camera arrays and robot state to the model contract. Then convert model output to a controller action. The starter images are small fixtures; the optional local rollout uses the saved policy and simulator.

**Notebook:** learner copy. Work through the cells in order.

In [ ]:
from pathlib import Path
import os, sys
WORKSPACE = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'manifests/act_references.yaml').is_file()), None)
if WORKSPACE is None:
    raise FileNotFoundError('Run this notebook inside the VLA course workspace')
sys.path.insert(0, str(WORKSPACE / 'src'))
HF_ROOT = WORKSPACE / '.local-data' / 'hf'
os.environ.setdefault('HF_HUB_CACHE', str(HF_ROOT / 'hub'))
os.environ.setdefault('HF_DATASETS_CACHE', str(HF_ROOT / 'datasets'))
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('HF_DATASETS_OFFLINE', '1')
print('Workspace:', WORKSPACE)


In [ ]:
from __future__ import annotations
import hashlib, json
import numpy as np
from vla_lab.teaching.adapters import camera_hwc_to_chw, assemble_libero_state, pack_observation, normalize_features, denormalize_features, controller_action


## Exercise 1: Convert one camera frame

**Purpose:** Prepare pixels for the model.

**Contract:** `camera_hwc_to_chw(image, *, channel_order='rgb') -> (3,H,W) float32`

Input is uint8 `(H,W,3)`. Output is contiguous RGB CHW float32 in `[0,1]`, rotated by 180 degrees as required by the pinned LIBERO processor.

**Method:** Flip both image axes, transpose HWC to CHW, and divide by 255. Reverse channels only when input is BGR.

**Pointers:** Check one red pixel after conversion; it must move to the opposite corner.

<details><summary>Detailed hint</summary>

NumPy slicing with `[::-1, ::-1]` rotates the image by 180 degrees.

</details>

Estimated time: about 15 minutes.

In [ ]:
def camera_hwc_to_chw(image, *, channel_order='rgb'):
    raise NotImplementedError('Exercise 1: convert camera layout, orientation, and scale')

### Check

In [ ]:
rgb = np.zeros((2, 3, 3), dtype=np.uint8)
rgb[0, 0] = [255, 0, 0]
converted = camera_hwc_to_chw(rgb)
assert converted.shape == (3, 2, 3), f'Expected CHW, got {converted.shape}'
assert np.allclose(converted[:, -1, -1], [1, 0, 0]), 'Pixel channels, 180-degree rotation, or scale are wrong'

### Use your function

In [ ]:
frame = np.zeros((64, 80, 3), dtype=np.uint8)
print('Model image:', camera_hwc_to_chw(frame).shape)

## Exercise 2: Assemble raw LIBERO state

**Purpose:** Convert nested robot readings into the eight values expected by ACT.

**Contract:** `assemble_libero_state(robot_state) -> (8,) float32`

Read `eef.pos` (3), `eef.quat` in XYZW order (4), and `gripper.qpos` (2). Return position, axis-angle rotation (3), and gripper qpos (2). Do not normalize these physical state values here.

**Method:** Normalize the quaternion. Convert its vector part to an axis-angle vector, then join the three fields.

**Pointers:** Identity rotation `[0,0,0,1]` becomes `[0,0,0]`.

<details><summary>Detailed hint</summary>

For quaternion `(x,y,z,w)`, use angle `2*atan2(norm([x,y,z]), w)` and multiply the unit axis by that angle.

</details>

Estimated time: about 15 minutes.

In [ ]:
def assemble_libero_state(robot_state):
    raise NotImplementedError('Exercise 2: map raw LIBERO robot fields to ACT state')

### Check

In [ ]:
robot = {'eef': {'pos': [1, 2, 3], 'quat': [0, 0, 0, 1]}, 'gripper': {'qpos': [-0.1, 0.1]}}
state = assemble_libero_state(robot)
assert state.shape == (8,) and np.allclose(state, [1, 2, 3, 0, 0, 0, -0.1, 0.1]), f'Wrong raw state mapping: {state}'

### Use your function

In [ ]:
print('Raw state:', state)

## Exercise 3: Assemble named inputs

**Purpose:** Keep camera order and state shape explicit.

**Contract:** `pack_observation(*, cameras, state, camera_order, channel_order='rgb') -> dict`

Each camera is uint8 HWC. State is finite float32 `(8,)`. Output keys match pinned ACT: `observation.images.image` and `observation.images.image2`.

**Method:** Convert each camera with Exercise 1; use the raw state from Exercise 2.

**Pointers:** Reject missing cameras instead of silently changing order.

<details><summary>Detailed hint</summary>

The first name in `camera_order` maps to `image`; the second maps to `image2`.

</details>

Estimated time: about 15 minutes.

In [ ]:
def pack_observation(*, cameras, state, camera_order, channel_order='rgb'):
    raise NotImplementedError('Exercise 3: assemble model inputs')

### Check

In [ ]:
frame = np.zeros((2, 3, 3), dtype=np.uint8)
batch = pack_observation(cameras={'front': frame, 'wrist': frame}, state=np.arange(8), camera_order=('front', 'wrist'))
assert batch['observation.images.image'].shape == (3, 2, 3), 'Pinned ACT image key or camera order is wrong'
assert batch['observation.state'].shape == (8,), 'State width must be eight'

### Use your function

In [ ]:
print('Input keys:', sorted(batch))

## Exercise 4: Normalize with saved statistics

**Purpose:** Put state or action features on the scale used for training.

**Contract:** `normalize_features(values, mean, std) -> array`

The last axis is the feature axis. `mean` and positive `std` are one-dimensional and match that axis.

**Method:** Compute `(values - mean) / std` with NumPy broadcasting.

**Pointers:** Use statistics fitted from training episodes only. The raw state from Exercise 2 has not been normalized yet.

<details><summary>Detailed hint</summary>

A `(2,8)` batch broadcasts against an `(8,)` mean.

</details>

Estimated time: about 15 minutes.

In [ ]:
def normalize_features(values, mean, std):
    raise NotImplementedError('Exercise 4: apply training statistics')

### Check

In [ ]:
normalized = normalize_features([[2, 4]], [1, 2], [1, 2])
assert np.allclose(normalized, [[1, 1]]), f'Unexpected normalized values: {normalized}'

### Use your function

In [ ]:
print('Normalized state:', normalize_features(np.arange(8), np.zeros(8), np.ones(8)))

## Exercise 5: Return predicted actions to physical units

**Purpose:** Undo the training action normalization before the controller receives a prediction.

**Contract:** `denormalize_features(values, mean, std) -> physical values`

The last axis is seven action features. Apply `values * std + mean` with saved action statistics.

**Method:** ACT predicts normalized actions. Convert them to controller units, then apply controller limits in Exercise 6.

**Pointers:** Clipping normalized predictions directly changes the model output scale.

<details><summary>Detailed hint</summary>

Use multiplication by standard deviation followed by addition of the training mean.

</details>

Estimated time: about 15 minutes.

In [ ]:
def denormalize_features(values, mean, std):
    raise NotImplementedError('Exercise 5: restore physical action scale')

### Check

In [ ]:
physical = denormalize_features([[1, -1]], [0.2, 0.0], [0.5, 2.0])
assert np.allclose(physical, [[0.7, -2.0]]), f'Wrong physical scale: {physical}'

### Use your function

In [ ]:
print('Physical action:', denormalize_features(np.zeros(7), np.zeros(7), np.ones(7)))

## Exercise 6: Bound an action in controller units

**Purpose:** Send one finite action to the simulator controller.

**Contract:** `controller_action(physical_action, *, lower=-1, upper=1) -> (7,) float32`

Input is a seven-value physical controller action. Validate shape and values, then clip within controller limits.

**Method:** Call only after denormalizing the model output with the saved training statistics.

**Pointers:** Clipping count is useful evidence; it does not repair a wrong scale.

<details><summary>Detailed hint</summary>

`np.clip` accepts scalar bounds and preserves the action shape.

</details>

Estimated time: about 15 minutes.

In [ ]:
def controller_action(physical_action, *, lower=-1.0, upper=1.0):
    raise NotImplementedError('Exercise 6: validate and bound a controller action')

### Check

In [ ]:
physical = denormalize_features([2, -2, 0, 0, 0, 0, 0], np.zeros(7), np.ones(7))
action = controller_action(physical)
assert action.shape == (7,) and np.allclose(action[:2], [1, -1]), 'Expected a bounded seven-value action'

### Use your function

In [ ]:
print('Controller action:', controller_action(np.zeros(7)))

## Larger experiment

Use the same local ACT policy, simulator state, and reset seed. Diagnose three supplied faults: camera order swapped, uint8 left unscaled, and normalized actions clipped before inverse normalization. Fix one fault at a time and record tensor shape/range, clipped count, success, and policy calls. Explore only states 0–9.

Set `RUN_LOCAL=True` only when the prepared local ACT checkpoint and LIBERO assets already exist. This cell loads the pinned observation processor, then uses your `pack_observation`, `normalize_features`, `denormalize_features`, and `controller_action` functions in callbacks passed to the shared real environment loop. It does not download files. The pinned ACT keys are `observation.images.image` and `observation.images.image2`. Keep recorded data at 10 Hz and simulator control at 20 Hz.

In [ ]:
# Small supplied faults run before any local model or simulator load.
fault_frames = {'front': np.zeros((2, 3, 3), dtype=np.uint8),
                'wrist': np.full((2, 3, 3), 255, dtype=np.uint8)}
swapped = pack_observation(cameras=fault_frames, state=np.zeros(8),
                           camera_order=('wrist', 'front'))
assert swapped['observation.images.image'].mean() == 1.0, 'Fault fixture did not swap camera order'
unscaled = fault_frames['wrist'].astype(np.float32)
assert unscaled.max() == 255.0, 'Fault fixture did not expose uint8 scale'
normalized = np.array([0.5], dtype=np.float32)
wrong_scale_action = controller_action(np.pad(normalized, (0, 6)))
correct_action = controller_action(denormalize_features(
    np.pad(normalized, (0, 6)), np.array([0.2] + [0.0] * 6), np.array([0.2] + [1.0] * 6)))
print('Faults: swapped front mean=', swapped['observation.images.image'].mean(),
      'unscaled max=', unscaled.max(), 'wrong-scale controller value=', wrong_scale_action[0],
      'denormalized controller value=', correct_action[0])

RUN_LOCAL = False
if RUN_LOCAL:
    import dataclasses
    import json
    import torch
    from vla_lab.teaching.live import find_workspace, load_prepared_policy
    from vla_lab.teaching.runtime import run_chunked_episode, prepare_libero_observation
    from vla_lab.sim.episode import open_libero_task

    root = find_workspace()
    policy, preprocessor, postprocessor, recipe = load_prepared_policy('act', root)
    policy.eval()
    device = next(policy.parameters()).device
    stats_file = json.loads((root / 'configs/data/stats/bowl_plate.json').read_text())
    state_stats = stats_file['modalities']['observation.state']
    action_stats = stats_file['modalities']['action']
    environment = open_libero_task('bowl_plate', 0, root / 'configs', root / '.local-data')
    learner_clipped_values = [0]
    def prepare(raw):
        batch = prepare_libero_observation(raw, preprocessor)
        raw_state = assemble_libero_state(raw['robot_state'])
        adapted = pack_observation(
            cameras={'front': raw['pixels']['image'], 'wrist': raw['pixels']['image2']},
            state=raw_state,
            camera_order=('front', 'wrist'),
        )
        batch['observation.images.image'] = torch.from_numpy(adapted['observation.images.image'][None]).to(device)
        batch['observation.images.image2'] = torch.from_numpy(adapted['observation.images.image2'][None]).to(device)
        batch['observation.state'] = torch.from_numpy(normalize_features(adapted['observation.state'], state_stats['mean'], state_stats['std'])[None]).to(device)
        return batch
    def predict(policy, batch):
        return policy.predict_action_chunk(batch)
    def to_controller(action):
        normalized = action.detach().cpu().numpy() if hasattr(action, 'detach') else action
        physical = denormalize_features(normalized, action_stats['mean'], action_stats['std'])
        learner_clipped_values[0] += int(np.count_nonzero((physical < -1.0) | (physical > 1.0)))
        return controller_action(physical)
    try:
        initial_observation, _info = environment.reset(seed=20260930)
        reference_batch = prepare_libero_observation(initial_observation, preprocessor)
        learner_batch = prepare(initial_observation)
        for key in ('observation.images.image', 'observation.images.image2', 'observation.state'):
            torch.testing.assert_close(learner_batch[key], reference_batch[key], atol=1e-5, rtol=1e-4,
                                       msg=lambda message: f'{key} adapter differs from pinned processor: {message}')
        with torch.no_grad():
            reference_chunk = policy.predict_action_chunk(reference_batch)
            learner_chunk = policy.predict_action_chunk(learner_batch)
            torch.testing.assert_close(learner_chunk, reference_chunk, atol=1e-5, rtol=1e-4,
                                       msg='Learner adapters changed ACT predictions')
            reference_physical = postprocessor(reference_chunk)
        learner_physical = denormalize_features(learner_chunk.detach().cpu().numpy(),
                                                 action_stats['mean'], action_stats['std'])
        np.testing.assert_allclose(learner_physical, reference_physical.detach().cpu().numpy(),
                                   atol=1e-5, rtol=1e-4)
        policy.reset()
        result = run_chunked_episode(environment, policy, prepare, predict, to_controller,
                                     max_steps=40, execute_steps=10, seed=20260930)
        result = dataclasses.replace(result, clipped_values=learner_clipped_values[0])
        print(result, 'learner action values clipped at physical-unit conversion=', learner_clipped_values[0])
    finally:
        environment.close()

## Record your result

Keep the inputs, every attempt, and the result table with the notebook. A failed run is a result and stays in the denominator.